# Customer Churn & Retention Analytics
### IBM SkillsBuild Data Analytics with AI Project

This notebook builds a complete end-to-end Machine Learning pipeline to analyze customer churn drivers and predict at-risk churn accounts.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from xgboost import XGBClassifier

print("Libraries successfully imported!")

## 1. Dataset Generation & Exploration

In [ ]:
np.random.seed(42)
n_samples = 1000

data = pd.DataFrame({
    'Tenure_Months': np.random.randint(1, 60, n_samples),
    'Monthly_Spend': np.round(np.random.uniform(20.0, 500.0, n_samples), 2),
    'Support_Tickets': np.random.randint(0, 10, n_samples),
    'Contract_Type': np.random.choice(['Month-to-Month', 'One-Year', 'Two-Year'], n_samples, p=[0.6, 0.25, 0.15]),
    'Paperless_Billing': np.random.choice(['Yes', 'No'], n_samples),
    'Payment_Method': np.random.choice(['Credit Card', 'Bank Transfer', 'E-Wallet'], n_samples)
})

# Logic for Churn
churn_prob = (
    0.35 * (data['Contract_Type'] == 'Month-to-Month').astype(int) +
    0.25 * (data['Support_Tickets'] > 4).astype(int) -
    0.30 * (data['Tenure_Months'] > 24).astype(int) +
    np.random.normal(0, 0.1, n_samples)
)
data['Churn'] = (churn_prob > 0.15).astype(int)

data.head()

## 2. Preprocessing & Model Building

In [ ]:
X = data.drop('Churn', axis=1)
y = data['Churn']

num_features = ['Tenure_Months', 'Monthly_Spend', 'Support_Tickets']
cat_features = ['Contract_Type', 'Paperless_Billing', 'Payment_Method']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first'), cat_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42))
])

model_pipeline.fit(X_train, y_train)
print("Model training completed.")

## 3. Evaluation & Visualization

In [ ]:
y_pred = model_pipeline.predict(X_test)
y_proba = model_pipeline.predict_proba(X_test)[:, 1]

print("=== Classification Report ===")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")